# Dados

In [3]:
# =============================================================================
# BIBLIOTECAS E MÓDULOS
# =============================================================================

import numpy as np
import pandas as pd
from common.utils import get_project_root
from custom_nlp.embeddings import load_embedding_model
from custom_nlp.tratamento_texto import (
    NormalizationStrategy,
    Preprocessor,
    StopwordsRemovalStrategy,
)
from sklearn.metrics.pairwise import cosine_similarity

# =============================================================================
# CONSTANTES
# =============================================================================

project_root_dir = get_project_root("classificador-cbo")
treated_data_dir = project_root_dir / "data/silver/cbo_sintese_perfil"

# -----------------------------------------------------------------------------
# Textos
# -----------------------------------------------------------------------------

eng_eletrico = "Engenheiro Eletricista com formação sólida e experiência em projetos, manutenção e operação de sistemas elétricos de baixa, média e alta tensão. Atuação em setores industriais, comerciais e/ou residenciais, com foco em eficiência energética, automação e segurança elétrica. Conhecimento aprofundado em normas técnicas (como a NBR 5410 e NR-10), elaboração de diagramas elétricos, uso de softwares especializados (AutoCAD, EPLAN, Matlab, etc.) e gestão de equipes multidisciplinares. Perfil analítico, proativo e comprometido com soluções técnicas seguras, sustentáveis e economicamente viáveis."
prof_geografia = "Professor de Geografia com sólida formação acadêmica e mais de [X anos] de experiência no ensino fundamental, médio e/ou superior. Especialista em metodologias ativas de aprendizagem, com foco em tornar o ensino da Geografia mais dinâmico, contextualizado e interdisciplinar. Ampla vivência em sala de aula, desenvolvendo projetos pedagógicos voltados à educação ambiental, geopolítica e cidadania. Excelente capacidade de comunicação, planejamento de aulas e avaliação de desempenho discente. Comprometido com a formação crítica dos alunos e com o uso de recursos tecnológicos para potencializar o aprendizado."
modelista = "Modelista com ampla experiência no desenvolvimento de moldes para confecção de peças do vestuário feminino, masculino e infantil. Domínio em modelagem plana, moulage e interpretação de fichas técnicas, com sólida capacidade de transformar croquis e ideias em peças-piloto de alto padrão. Conhecimento técnico em tecidos, caimentos e encaixes, além de domínio de softwares de modelagem como Audaces, Gerber e Lectra. Perfil detalhista, criativo e comprometido com a qualidade, prazos e processos de produção em ateliês ou confecções industriais."

# -----------------------------------------------------------------------------
# Subgrupos Principais
# -----------------------------------------------------------------------------
subgrupos_principais = pd.read_csv(
    treated_data_dir / "subgrupo_principal.csv", sep="\t"
)

# -----------------------------------------------------------------------------
# Pré-tratamento utilizados
# -----------------------------------------------------------------------------

corpus_referencia = subgrupos_principais.loc[:, "perfilocupacional"]

# Fazendo o pré-processamento textual
# 1. Normalizando
preprocessador = Preprocessor(strategy=NormalizationStrategy)
corpus_referencia = np.vectorize(pyfunc=preprocessador.apply)(corpus_referencia)

# 2. Removendo stopwords
preprocessador.set_strategy(strategy=StopwordsRemovalStrategy, language="portuguese")
corpus_referencia = np.vectorize(pyfunc=preprocessador.apply)(corpus_referencia)


def trata_texto(texto: str) -> str:
    preprocessador = Preprocessor(strategy=NormalizationStrategy)

    # 1. Normalizando
    texto_tratado = preprocessador.apply(text=texto)

    # 2. Removendo stopwords
    preprocessador.set_strategy(
        strategy=StopwordsRemovalStrategy, language="portuguese"
    )
    texto_tratado = preprocessador.apply(text=texto_tratado)

    return texto_tratado

## Usando CountVect

In [4]:
embedder = load_embedding_model("tfidfVect_familiaCbo_normalizado_s_stopword")

vetores_referencia = embedder.transform(raw_documents=corpus_referencia)
vetor_texto = embedder.transform(raw_documents=[trata_texto(prof_geografia)])

similaridades = []
for subgrupo in vetores_referencia:
    similaridades.append(cosine_similarity(X=subgrupo, Y=vetor_texto).item())

argmax = np.argmax(similaridades)
subgrupos_principais.iloc[argmax, -1]

'PROFISSIONAIS DO ENSINO                                                                                                           '

## Usando Word2Vec

In [ ]:
w2v = load_embedding_model("w2v_familiaCbo_normalizado_s_stopword")

In [6]:
vetores_subgrupos = [w2v.wv.get_mean_vector(texto) for texto in corpus_referencia]

In [10]:
a = w2v.wv.cosine_similarities(
    vector_1=w2v.wv.get_mean_vector(trata_texto(prof_geografia).split()),
    vectors_all=vetores_subgrupos,
).argmax()
subgrupos_principais.iloc[a, -1]

'MEMBROS SUPERIORES E DIRIGENTES DO PODER PÚBLICO                                                                                  '